In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import matplotlib.pyplot as plt
import seaborn as sns

# USA Vehicle Sales Data Analysis

## What 
The Vehicle Sales and Market Trends Dataset provides a collection of information about the sales transactions of various vehicles from January of 2014 to 21 July 2015 in USA.

## Who
Car Dealership trying to see market trends for vehicle acquisition and deciding on a pricing strategy

##Data Gaps:
- Since maximum year of production included in this data is 2015, this analysis will use 2015 as the "now" to compare age of vehicles
- The dataset does not have transactions for August - November 2014 so the time-series analysis for this gap is unaccounted for

## Why: 
### 1. Performance & Valuation (MMR vs. Actual)
This category focuses on the relationship between the MMR and what actually happens in real life.
- Average Price Variance: How big is the average difference between Manheim Market Report (MMR) price and the actual selling price?
- Brand Performance: Which brands (make) are sold with a higher selling price, and which are sold lower than MMR?
- Age-Based Performance: How do younger vs. older cars perform? How does the MMR-to-selling price gap change as a car ages?
- Oversupply vs. Undersupply: Are there specific models that consistently sell above or below MMR due to supply issues?

### 2. Vehicle Attributes & Value Drivers
This category looks at the specific physical characteristics of a car and how they move the needle on its final price.
- Condition & Aesthetics: How do condition, color, and interior quality affect the selling price?
- Usage: How does the odometer (mileage) specifically impact the final value?
- Optimal Profile: What is the "ideal" vehicle profile (combination of age, color, mileage, and make) for maximum selling value?

# Insights for Car Dealerships

## Performance & Valuation 
* Younger cars (0-3 years) exhibit the lowest selling price from MMR, as vehicles age,
* Suzuki, Acura, Subaru, GMC, Audi cars currently represent "value stability", these brands consistenly trading at or above fair market value, suggesting reliable demand.
* Cars made by Lincoln, Jaguar, Saturn, Mercury, and Pontiac are severely underperforming. For Jaguar, the premium maintenance costs act as a barrier, while for Saturn, Mercury, and Pontiac, the lack of official service channels and parts scarcity since their discontinuation creates a significant psychological discount for buyers.
* Since every body type is currently underperforming relative to MMR, this confirms that MMR is a lagging indicator. In the current market, "What" the car is (SUV vs. Sedan) matters less than "How" it is and "Who" made it.
* Several models are currently oversupplied, these includes Pontiac	Grand Prix, Cadillac DeVille, Chrysler PT Cruiser, Chrysler	Sebring, and Chevrolet	Aveo. These models are typically sold 5-10% lower than their MMR. However, there are no common car models that are now significantly undersupplied and should specifically be acquired for dealers.
* State is important for buying or selling cars. Cars sold in Tennessee, North Virginia, California, North Carolina, and Washington typically sell for 1-2% more. Meanwhile, cars sold in Utah, Hawaii, Maryland, New York, and Massachussetts tend to be sold 5-9% under the MMR

## Vehicle Attributes & Value Drivers
* The single most deciding actual price driver is the odometer (mileage), cars with over 90.000 Km mileage loses 2-5% of its market value.
* Cars with condition under 4.0 see price drops of up to 17% relative to MMR, while cars with condition of 4.0 and above tend to be priced exactly or slightly above its MMR.
* Top performers for exterior color are Lime (+0.029), Off-white (+0.016), and Yellow (+0.015), suggesting these distinct colors are in high demand or represent rare enthusiast specifications. White and Yellow are near fair market value, showing reliability and low risk-choices. Meanwhile, Pink is a significant outlier, selling almost 7% below MMR. Pink, Green, and Turquoise underperdorm and suggests narrower buyer pool.
* For car interiors, Yellow, Orange, White, and Burgundy act as lead value drivers (0.1-2% price increase from MMR), likely due to their proximity to "premium" branding. Off-white and Blue are the least desirable in this dataset. 

# Recommendations
* There is a significant profit authority in geography. Sourcing a vehicle in an underperforming region like Massachusetts and selling it in a high-demand region like Washington represents a theoretical 6–10% margin based purely on location. Other premium markets include Tennessee, North Virginia, California, and North Carolina.
* Avoid discontinued  (such as Saturn, Mercury, or Pontiac) unless they are bought at at least 15% below MMR to account for their "service channel" discontinuation.
* Prioritize <90.000 Km mileage, high-stability brands like Acura or Suzuki, and ensure the unit condition is 4.0 or higher when acquiring new cars.
* The "Ideal Unit" for maximum return is defined as a 0–3 year old Suzuki, Acura, or Subaru with under 90,000 Km of mileage and a condition grade of 4.0 or higher. For peak enthusiast demand and premium pricing, the ideal aesthetic combination features a Lime exterior paired with a Yellow interior.

# DATA CLEANING

In [ ]:
df = pd.read_csv('/kaggle/input/vehicle-sales-data/car_prices.csv')

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Number of rows:",df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
df['saledate'] = pd.to_datetime(
    df['saledate'], 
    format='%a %b %d %Y %H:%M:%S', 
    exact=False, 
    errors='coerce'
)

In [ ]:
print(df['saledate'].min(), "--",  df['saledate'].max())

In [ ]:
all_months = pd.period_range(start=df['saledate'].min(), end=df['saledate'].max(), freq='M')
existing_months = df['saledate'].dt.to_period('M').unique()
missing_months = all_months.difference(existing_months)

print(f"Missing Months ({len(missing_months)} total):")
for month in missing_months:
    print(month)

In [ ]:
print(df.isnull().sum() / len(df) * 100)

In [ ]:
df = df.dropna(subset=['mmr', 'sellingprice', 'make', 'body', 'model'])

In [ ]:
df = df.drop('transmission', axis=1)

In [ ]:
df['condition']= df.groupby(['make', 'year'])['condition'].transform(
    lambda x: x.fillna(x.median())
)

df['odometer'] = df.groupby(['make', 'year'])['odometer'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
print(df.isnull().sum()) 

In [ ]:
df = df.dropna(subset=['condition', 'color', 'interior'])
df = df[df['interior'] != '—']

In [ ]:
print(df.isnull().sum()) 

###  Date & Seasons

In [ ]:
df['saledate'].head(3)

In [ ]:
df['sale_year'] = df['saledate'].dt.year
df['sale_month'] = df['saledate'].dt.month

In [ ]:
conditions = [
    (df['sale_month'] >= 3) & (df['sale_month'] <= 5),  
    (df['sale_month'] >= 6) & (df['sale_month'] <= 8), 
    (df['sale_month'] >= 9) & (df['sale_month'] <= 11), 
    (df['sale_month'] == 12) | (df['sale_month'] <= 2) 
]
seasons = ['Spring', 'Summer', 'Fall', 'Winter']

In [ ]:
df['sale_season'] = np.select(conditions, seasons, default='Unknown')

In [ ]:
df[['sale_year', 'sale_month', 'sale_season']].head(3)

In [ ]:
df.nunique()

In [ ]:
print(df['sale_month'].value_counts().sort_index())

In [ ]:
print(df['year'].min(), "-",  df['year'].max())

In [ ]:
df['age'] = 2015 - df['year']

In [ ]:
df['condition'] = pd.cut(df['condition'], 
                               bins=[0, 10, 20, 30, 40, 50],
                               labels=[1, 2, 3, 4, 5])

In [ ]:
df['body'].unique()

In [ ]:
df['body'] = df['body'].str.lower().str.strip()

body_map = {
    'sedan': 'Sedan', 'g sedan': 'Sedan', 
    
    'coupe': 'Coupe', 'g coupe': 'Coupe', 'elantra coupe': 'Coupe', 
    'genesis coupe': 'Coupe', 'cts coupe': 'Coupe', 'koup': 'Coupe', 
    'cts-v coupe': 'Coupe', 'q60 coupe': 'Coupe', 'g37 coupe': 'Coupe',
    
    'suv': 'SUV', 
    
    'crew cab': 'Pickup Truck', 'double cab': 'Pickup Truck', 'king cab': 'Pickup Truck', 
    'supercrew': 'Pickup Truck', 'extended cab': 'Pickup Truck', 'supercab': 'Pickup Truck', 
    'regular cab': 'Pickup Truck', 'quad cab': 'Pickup Truck', 'crewmax cab': 'Pickup Truck', 
    'access cab': 'Pickup Truck', 'club cab': 'Pickup Truck', 'xtracab': 'Pickup Truck', 
    'mega cab': 'Pickup Truck', 'cab plus 4': 'Pickup Truck', 'cab plus': 'Pickup Truck',
    'regular-cab': 'Pickup Truck',
    
    'convertible': 'Convertible', 'g convertible': 'Convertible', 
    'g37 convertible': 'Convertible', 'q60 convertible': 'Convertible', 
    'beetle convertible': 'Convertible', 'granturismo convertible': 'Convertible',
    
    'minivan': 'Van/Minivan', 'van': 'Van/Minivan', 'e-series van': 'Van/Minivan', 
    'promaster cargo van': 'Van/Minivan', 'transit van': 'Van/Minivan',
    
    'wagon': 'Wagon/Hatchback', 'hatchback': 'Wagon/Hatchback', 
    'cts wagon': 'Wagon/Hatchback', 'tsx sport wagon': 'Wagon/Hatchback', 
    'cts-v wagon': 'Wagon/Hatchback'
}

df['body'] = df['body'].map(body_map).fillna('Other')

In [ ]:
df['body'].unique()

# Exploratory Data Analysis

## Market Performance & Valuation (MMR vs Actual)


In [ ]:
monthly_stats = df.groupby(df['saledate'].dt.to_period('M'))['price_ratio'].agg(['mean', 'median']).sort_index().reset_index()
monthly_stats['label'] = monthly_stats['saledate'].dt.strftime('%b\n%Y')

make_stats = df.groupby('make')['price_ratio'].agg(['median', 'count']).reset_index()
make_stats = make_stats[make_stats['count'] > 1000]
top_5_makes = make_stats.sort_values(by='median', ascending=False).head(5)
bottom_5_makes = make_stats.sort_values(by='median', ascending=True).head(5)
make_comparison = pd.concat([bottom_5_makes, top_5_makes]).sort_values(by='median')

age_trend = df.groupby([df['saledate'].dt.to_period('M'), 'age_category'], observed=True)['price_ratio'].median().reset_index()
age_trend['month_year'] = age_trend['saledate'].dt.strftime('%b %Y')

model_stats = df.groupby(['make', 'model'], observed=True)['price_ratio'].agg(['median', 'count']).reset_index()
reliable_models = model_stats[model_stats['count'] > 500]
extreme_models = pd.concat([reliable_models.nsmallest(5, 'median'), reliable_models.nlargest(5, 'median')])
extreme_models['full_name'] = extreme_models['make'] + " " + extreme_models['model']

state_stats = df.groupby('state')['price_ratio'].agg(['median', 'count']).reset_index()
state_impact = pd.concat([state_stats[state_stats['count'] > 1000].sort_values('median').head(5), 
                         state_stats[state_stats['count'] > 1000].sort_values('median').tail(5)]).sort_values('median')

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(15, 45))
sns.set_style("whitegrid")

x_range = range(len(monthly_stats))
axes[0].plot(x_range, monthly_stats['mean'], marker='o', markersize=8, color='#3498db', label='Mean', linewidth=3)
axes[0].plot(x_range, monthly_stats['median'], marker='s', markersize=8, color='#27ae60', label='Median', linewidth=3)
axes[0].axhline(1.0, color='#e74c3c', linestyle='--')
axes[0].set_title('I. Monthly Market Trend: Mean vs. Median Price Ratio', fontsize=18, fontweight='bold', loc='left')
axes[0].set_xticks(x_range)
axes[0].set_xticklabels(monthly_stats['label'])
for i in x_range:
    axes[0].text(i, monthly_stats['mean'].iloc[i] + 0.005, f"{monthly_stats['mean'].iloc[i]:.3f}", ha='center', color='#2980b9', fontweight='bold')

make_comparison['deviation'] = make_comparison['median'] - 1.0
axes[1].bar(make_comparison['make'], make_comparison['deviation'], color=['#e74c3c' if x < 0 else '#2ecc71' for x in make_comparison['deviation']], edgecolor='black')
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].set_title('II. Brand Performance: Top 5 vs. Bottom 5 Makes', fontsize=18, fontweight='bold', loc='left')
for i, val in enumerate(make_comparison['deviation']):
    axes[1].text(i, val + (0.005 if val > 0 else -0.015), f"{val+1:.3f}", ha='center', fontweight='bold')

sns.lineplot(data=age_trend, x='month_year', y='price_ratio', hue='age_category', marker='o', palette='viridis', ax=axes[2], linewidth=3)
axes[2].axhline(1.0, color='#e74c3c', linestyle='--')
axes[2].set_title('III. Market Trends by Age Group Over Time', fontsize=18, fontweight='bold', loc='left')
axes[2].tick_params(axis='x', rotation=30)


palette_iv = {'Undersupply': '#2ecc71', 'Oversupply': '#e74c3c'}
extreme_models['Group'] = np.where(extreme_models['median'] > 1.0, 'Undersupply', 'Oversupply')
sns.scatterplot(data=extreme_models, x='count', y='median', hue='Group', s=400, palette=palette_iv, ax=axes[3], edgecolor='black', alpha=0.8)
axes[3].axhline(1.0, color='#34495e', linestyle='--')
axes[3].set_title('IV. Model Extremes: Supply (Volume) vs. Demand (Ratio)', fontsize=18, fontweight='bold', loc='left')
for i in range(len(extreme_models)):
    axes[3].text(extreme_models['count'].iloc[i], extreme_models['median'].iloc[i] + 0.005, extreme_models['full_name'].iloc[i], ha='center', fontweight='bold')


state_impact['dev'] = state_impact['median'] - 1.0
axes[4].barh(state_impact['state'], state_impact['dev'], color=['#e74c3c' if x < 0 else '#27ae60' for x in state_impact['dev']], edgecolor='black')
axes[4].axvline(0, color='black', linewidth=1.5)
axes[4].set_title('V. Regional Arbitrage: Where Prices Exceed MMR', fontsize=18, fontweight='bold', loc='left')
axes[4].set_xlim(state_impact['dev'].min() - 0.02, state_impact['dev'].max() + 0.02)
for i, row in enumerate(state_impact.itertuples()):
    align = 'right' if row.dev < 0 else 'left'
    axes[4].text(row.dev + (-0.002 if row.dev < 0 else 0.002), i, f'{row.dev:+.1%}', va='center', fontweight='bold', ha=align)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.suptitle('Macro-Market Drivers & Inventory Economics', fontsize=26, fontweight='bold', y=0.995)
plt.show()

## Vehicle Attributes & Value Drivers 

In [ ]:
df['cond_grade'] = df['condition'].round()
cond_trend = df.groupby('cond_grade', observed=False)['price_ratio'].median().reset_index()

df['mileage_bin'] = pd.cut(df['odometer'], 
                            bins=[0, 30000, 60000, 90000, 120000, 1000000],
                            labels=['0-30k', '30-60k', '60-90k', '90-120k', '120k+'])
mileage_impact = df.groupby('mileage_bin', observed=True)['price_ratio'].median().reset_index()

body_impact = df.groupby('body').agg({'price_ratio': 'median', 'vin': 'count'}).reset_index()
body_impact.columns = ['body', 'price_ratio', 'volume']
body_impact = body_impact[body_impact['volume'] > 100].sort_values('price_ratio', ascending=True)

color_all = df.groupby('color')['price_ratio'].median().sort_values(ascending=False)
color_impact = pd.concat([color_all.head(5), color_all.tail(5)]).reset_index()
color_impact['deviation'] = color_impact['price_ratio'] - 1.0

interior_all = df.groupby('interior')['price_ratio'].median().sort_values(ascending=False)
interior_impact = pd.concat([interior_all.head(5), interior_all.tail(5)]).reset_index()
interior_impact['deviation'] = interior_impact['price_ratio'] - 1.0

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 40))
sns.set_style("whitegrid")

axes[0].plot(cond_trend['cond_grade'], cond_trend['price_ratio'], marker='o', markersize=12, linewidth=4, color='#2980b9', zorder=3)
axes[0].axhline(1.0, color='#e74c3c', linestyle='--', zorder=2)
axes[0].set_title('I. Condition vs. Valuation Accuracy', fontsize=18, fontweight='bold', loc='left')
axes[0].set_ylabel('Median Price/MMR')
for x, y in zip(cond_trend['cond_grade'], cond_trend['price_ratio']):
    axes[0].text(x, y + 0.005, f'{y:.3f}', ha='center', fontweight='bold', fontsize=11)

bar_colors_mil = ['#27ae60' if x >= 1.0 else '#e74c3c' for x in mileage_impact['price_ratio']]
bars_ii = axes[1].bar(mileage_impact['mileage_bin'], mileage_impact['price_ratio'], color=bar_colors_mil, edgecolor='black', alpha=0.8)
axes[1].axhline(1.0, color='#34495e', linestyle='--')
axes[1].set_ylim(0.85, 1.15) 
axes[1].set_title('II. Mileage Impact: Psychological Thresholds', fontsize=18, fontweight='bold', loc='left')
for bar in bars_ii:
    h = bar.get_height()
    var = (h - 1.0) * 100
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 0.008, f'{h:.3f}\n({var:+.1f}%)', ha='center', fontweight='bold', fontsize=11)

axes[2].axvspan(0.99, 1.0, color='#fdf2f2', alpha=0.5) 
axes[2].axvspan(1.0, 1.005, color='#f2fdf2', alpha=0.5)
colors_body = ['#e74c3c' if x < 1.0 else '#27ae60' for x in body_impact['price_ratio']]
axes[2].barh(body_impact['body'], body_impact['price_ratio'], color=colors_body, edgecolor='black', height=0.6, zorder=3)
axes[2].axvline(1.0, color='black', linestyle='-', linewidth=3, zorder=4)
axes[2].set_xlim(0.993, 1.006)
axes[2].set_title('III. Body Type: Performance Target Gauge', fontsize=18, fontweight='bold', loc='left')
for i, v in enumerate(body_impact['price_ratio']):
    axes[2].text(v + 0.0001, i, f'{v:.4f} (n={body_impact["volume"].iloc[i]:,})', va='center', fontweight='bold', fontsize=11)

color_sorted = color_impact.sort_values('deviation')
bars_iv = axes[3].barh(color_sorted['color'], color_sorted['deviation'], color=['#e74c3c' if x < 0 else '#27ae60' for x in color_sorted['deviation']], edgecolor='black')
axes[3].axvline(0, color='black', linewidth=1.5)
axes[3].set_xlim(-0.12, 0.12) 
axes[3].set_title('IV. Exterior Color: Premium vs. Discount', fontsize=18, fontweight='bold', loc='left')
for bar in bars_iv:
    w = bar.get_width()
    offset = -0.005 if w < 0 else 0.005
    align = 'right' if w < 0 else 'left'
    axes[3].text(w + offset, bar.get_y() + bar.get_height()/2, f'{w:+.3f}', va='center', ha=align, fontweight='bold', fontsize=11)

int_sorted = interior_impact.sort_values('deviation')
bars_v = axes[4].barh(int_sorted['interior'], int_sorted['deviation'], color=['#e74c3c' if x < 0 else '#27ae60' for x in int_sorted['deviation']], edgecolor='black')
axes[4].axvline(0, color='black', linewidth=1.5)
axes[4].set_xlim(-0.08, 0.08) 
axes[4].set_title('V. Interior Color: Value Impact', fontsize=18, fontweight='bold', loc='left')
for bar in bars_v:
    w = bar.get_width()
    offset = -0.003 if w < 0 else 0.003
    align = 'right' if w < 0 else 'left'
    axes[4].text(w + offset, bar.get_y() + bar.get_height()/2, f'{w:+.3f}', va='center', ha=align, fontweight='bold', fontsize=11)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.suptitle('The Dealer Advantage: Key Market Value Drivers', fontsize=26, fontweight='bold', y=0.99)
plt.show()

# Feature Importance analysis using a Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

target_features = ['make', 'state', 'color', 'interior']
ohe_features = ['body']
numeric_features = ['condition', 'odometer', 'age']

X = df[['condition', 'odometer', 'color', 'interior', 'state', 'age', 'body', 'make']]
y = df['price_ratio']

preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(target_type='continuous'), target_features),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_features),
        ('num', 'passthrough', numeric_features)
    ],
    verbose_feature_names_out=False
)

X_encoded = preprocessor.fit_transform(X, y)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_encoded, y)

feature_names = preprocessor.get_feature_names_out()
importances = rf.feature_importances_

In [ ]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

importance_df['Feature'] = importance_df['Feature'].str.replace(r'^(target__|ohe__|num__)', '', regex=True)
top_5_df = importance_df.head(5)

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

sns.barplot(
    x='Importance', 
    y='Feature', 
    data=top_5_df, 
    hue='Feature', 
    palette='viridis', 
    legend=False
)

plt.title('Top 5 Market Value Drivers (Random Forest)', fontsize=16, fontweight='bold', loc='left')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature Name', fontsize=12)
for i, val in enumerate(top_5_df['Importance']):
    plt.text(val + 0.005, i, f'{val:.4f}', va='center', fontweight='bold', fontsize=11)

plt.xlim(0, top_5_df['Importance'].max() * 1.15)

plt.tight_layout()
plt.savefig('top_5_drivers.png')
plt.show()